#   [AWS Step Functions Simplified by Knowledge Amplifier](https://www.youtube.com/playlist?list=PLjfRmoYoxpNoahLvGnz_vJnOZ2FBQhaVu)

In [ ]:
import boto3
import botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile, subprocess, shutil, requests
from requests_aws4auth import AWS4Auth
from pathlib import Path
from datetime import date

import s3, iam, lf, glue, lambdafn as lfn, sns, eventbridge as event

from ads.utils import red

In [ ]:
from dotenv import load_dotenv

load_dotenv("../env")
AWS_ALL_IN_ONE_SG = os.environ["AWS_ALL_IN_ONE_SG"]
ACCOUNT_ID = os.environ["AWS_ACCOUNT_ID_ROOT"]
REGION = os.environ["AWS_DEFAULT_REGION"]
VPC_ID = os.environ["AWS_DEFAULT_VPC"]
SECURITY_GROUP_ID = os.environ["AWS_DEFAULT_SG_ID"]
SUBNET_IDS = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID = SUBNET_IDS[0]
AWS_INSTANCE_ID_JMASTER = os.environ["AWS_INSTANCE_ID_JMASTER"]
AWS_INSTANCE_ID_JAGENT = os.environ["AWS_INSTANCE_ID_JAGENT"]
AWS_DEFAULT_IMAGE_ID = os.environ["AWS_DEFAULT_IMAGE_ID"]
AWS_DEFAULT_KEY_PAIR_NAME = os.environ["AWS_DEFAULT_KEY_PAIR_NAME"]
AWS_DEFAULT_INSTANCE_TYPE = os.environ["AWS_DEFAULT_INSTANCE_TYPE"]
AWS_DEFAULT_IMAGE_ID = os.environ["AWS_DEFAULT_IMAGE_ID"]
AMAZON_LINUX_AMI_ID = os.environ["AMAZON_LINUX_AMI_ID"]


In [ ]:
sts_client           = boto3.client('sts')
iam_client           = boto3.client('iam')
s3_client            = boto3.client('s3')
glue_client          = boto3.client('glue')
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
lfn_client           = boto3.client('lambda')
sfn_client           = boto3.client('stepfunctions')
logs_client          = boto3.client('logs')
events_client        = boto3.client('events')

apigateway_client    = boto3.client('apigateway', region_name=REGION)

### [BeABetterDev: Build a Serverless Workflow with AWS Step Functions](https://www.youtube.com/watch?v=DFSko_sLyMM&list=PL9nWRykSBSFjrGfOfH5wJ6Gl56N746obR&index=10&t=268s)

### [Invoking State Machine with CloudWatch](https://www.youtube.com/watch?v=22yRNLm6BbQ&list=PLjfRmoYoxpNoahLvGnz_vJnOZ2FBQhaVu&index=4)

### [AWS Tutorials - Using Amazon API Gateway in AWS Step Functions](https://www.youtube.com/watch?v=dYpc8O-2JuM)
- [lab](https://aws-dojo.com/excercises/excercise28/)

### [Adding manual approval step to the AWS Step Function workflow](https://www.youtube.com/watch?v=URhLte0NJJU&list=PLjfRmoYoxpNoahLvGnz_vJnOZ2FBQhaVu&index=13)

<div style="text-align: center"><img src="./architectural_diagram.png" length="500p" height="300p"></img></div>

Step Function Code:
--------------------------

```json
{
  "Comment": "A description of my state machine",
  "StartAt": "SQS SendMessage",
  "States": {
    "SQS SendMessage": {
      "Type": "Task",
      "Resource": "arn:aws:states:::sqs:sendMessage.waitForTaskToken",
      "Parameters": {
        "MessageBody": {
          "input.$": "$",
          "MyTaskToken.$": "$$.Task.Token"
        },
        "QueueUrl": "{Put the SQS Queue URL here}"
      },
      "Next": "Choice"
    },
    "Choice": {
      "Type": "Choice",
      "Choices": [
        {
          "Variable": "$.body",
          "StringEquals": "Approved",
          "Next": "Success"
        }
      ],
      "Default": "Fail"
    },
    "Success": {
      "Type": "Succeed"
    },
    "Fail": {
      "Type": "Fail"
    }
  }
}
```

Step Function Sample Input:
-----------------------------

```json
{
  "Manager Mail Address": "{}",
  "Employee Name":"{}"
}
```

Callback Lambda Code:
-----------------------------

```python
import json
import boto3
import time
import urllib

client = boto3.client("ses")

def lambda_handler(event, context):
    main_message=json.loads(event['Records'][0]['body'])
    print("Main Message Part : {}".format(main_message))
    
    step_fucntion_input=main_message['input']
    
    manager_main_address=step_fucntion_input['Manager Mail Address']
    employee_to_be_promoted=step_fucntion_input['Employee Name']

    
    task_token=main_message['MyTaskToken']
    print("The task token is : {}".format(task_token))
    task_token_encode=urllib.parse.quote(task_token)
    body = """
                 Hi,<br>
                     {} has been nominated for promotion!<br />.
                     
                 Can you please approve:<br />
                 
                 {Put the API Invoke URL here}/approve?TaskToken={}<br />
                 
                 Or reject:<br />
                 
                 {Put the API Invoke URL here}/reject?TaskToken={}
         """.format(employee_to_be_promoted, task_token_encode,task_token_encode)
         
    message = {"Subject": {"Data": 'Your Approval Needed for Promotion!'}, "Body": {"Html": {"Data": body}}}
    
    response = client.send_email(Source = manager_main_address, Destination = {"ToAddresses": [manager_main_address]}, Message = message) 
    
    print("The mail is sent successfully")
```


Approve Handler:
------------------------

```python
import json
import boto3
import time

client = boto3.client('stepfunctions')

def lambda_handler(event, context):
    task_token=event['queryStringParameters']['TaskToken']
    print(task_token)
    response = client.send_task_success(
    taskToken=task_token,
    output=json.dumps({'body':'Approved'})
    )
```
	
Reject Handler:
------------------

```python
import json
import boto3
import time

client = boto3.client('stepfunctions')

def lambda_handler(event, context):
    task_token=event['queryStringParameters']['TaskToken']
    response = client.send_task_success(
    taskToken=task_token,
    output=json.dumps({'body':'Rejected'})
    )
```

### [Automating EMR Serverless Workload |Creating|Submitting | Destroying EMR Cluster using Step Function](https://www.youtube.com/watch?v=V7bFwXBN5xc)
-   [code](https://github.com/soumilshah1995/Automating-EMR-Serverless-Workload-Creating-Submitting-Destroying-EMR-Cluster-using-Step-Funct)